In [0]:
SELECT
  scraped_at ,
  product_id,
  retailer,
  get_json_object(raw_data, '$.sku') AS sku,
  CASE
    WHEN get_json_object(raw_data, '$.name') IS NULL THEN get_json_object(raw_data, '$.title')
    ELSE get_json_object(raw_data, '$.name')
  END as name,
  get_json_object(raw_data, '$.brand') AS brand,
  get_json_object(raw_data, '$.main_category') AS main_category,
  get_json_object(raw_data, '$.sub_category') AS sub_category,
  get_json_object(raw_data, '$.list_price') AS list_price,
  get_json_object(raw_data, '$.cash_price') AS cash_price,
  get_json_object(raw_data, '$.stock') AS is_in_stock,
  get_json_object(raw_data, '$.installments') AS installments,
  get_json_object(raw_data, '$.description') AS description,
  get_json_object(raw_data, '$.specifications') AS specifications,
  get_json_object(raw_data, '$.rating') AS rating
FROM
  products.bronze_scraped_products
ORDER BY
  scraped_at DESC
LIMIT 10;

SELECT 
  product_id,
  retailer,
  scraped_at,
  COUNT(*) as cnt
FROM workspace.products.bronze_scraped_products
GROUP BY 1,2,3
HAVING COUNT(*) > 1;

WITH last_two_price_updates AS (
  SELECT
    product_id,
    retailer,
    get_json_object(raw_data, '$.list_price') as list_price,
    get_json_object(raw_data, '$.cash_price') as cash_price,
    scraped_at,
    ROW_NUMBER() OVER (PARTITION BY product_id, retailer ORDER BY scraped_at DESC) as rn
  FROM
    products.bronze_scraped_products
)
SELECT
  product_id,
  retailer,
  list_price,
  cash_price,
  scraped_at
FROM last_two_price_updates
WHERE rn <= 2
ORDER BY RIGHT(product_id, 3), retailer, scraped_at DESC;